In [0]:
# Index
# all helpers
# get Reference Data
# main get Reference Data 
# get Fligt data 

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS data_Catalog")

spark.sql("CREATE SCHEMA IF NOT EXISTS data_Catalog.bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS data_Catalog.bronze.bronze_volume")

spark.sql("USE CATALOG data_Catalog")
spark.sql("USE SCHEMA bronze")
schema_name = "bronze"

full_table_name = f"lufthansa.{schema_name}.airport"

# Drop the table if it exists
spark.sql(f"DROP TABLE IF EXISTS `{full_table_name}`")
DATA_PATH = "/Volumes/data_Catalog/bronze_volume/airport"

In [ ]:
import requests
import json
import os
from datetime import datetime

from pathlib import Path
from typing import Dict, Any, List, Optional
import uuid
import time
from urllib.parse import urlparse, parse_qs
import sys

def update_offset(recordLimit: int, offset: int, TotalCount: int)-> str: 
        if recordLimit < TotalCount:
            offset = offset + recordLimit
            return offset
        return offset 
    
def timeout_api_restriction(responseCode: int )->bool:
        if responseCode == 503 or responseCode == 504 or responseCode == 429:
            print(f"api restriction: {responseCode}")
            time.sleep(4)
            return True
        return False
        
def versioning_fileNames(filename: str, offset:int ) -> str:
        """ differ by milliseconds , offset and unique key e.g. airports_2026-03-12-15-34-21-482_off200_a1f9c3.json"""
        unique_key = uuid.uuid4().hex[:6]
        timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S-%f")[:-3]
        versioned_filename = f"{filename}_{timestamp}_{offset}_{unique_key}"
        return versioned_filename
    
def loop_until_data_pool_finished(Totaldata: int, recordLimit: int)->bool:
        if Totaldata > recordLimit:
            return True
        else:
            False

def reset_timeout_rounds(timeout_rounds:int)->int:
    timeout_rounds = 0
    return timeout_rounds

def save_json_locally(
    json_data: Any,
    base_filename: str,
    offset: int,
    local_folder: Optional[str] = None,
    ) -> None:
    """
    save json localy 
    """
    versioned_filename = versioning_fileNames(base_filename, offset)

    if local_folder is not None:
        os.makedirs(local_folder, exist_ok=True)
        file_path = os.path.join(local_folder, versioned_filename)
    else:
        file_path = versioned_filename

    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(json_data, file, indent=2, ensure_ascii=False)

    print(file_path)
    print(f"saved locally: {file_path}")

def save_in_Notebooks(FileName_base,json_data, offset)-> None:
    versioned_filename = versioning_fileNames(FileName_base, offset)
    with open (versioned_filename, "w") as file:
        json.dump(json_data, file, indent=2)
    print(f"{versioned_filename} saved")


def get_next_endpoint_from_response(json_data: dict, meta_data_key: str) -> Optional[str]:
    """
    read api respond json file for link @Rel == 'next' and
    changes it to a Endpoint for proxy 
    """
    counter = 0
    airport_resource = json_data.get(meta_data_key, {})
    meta = airport_resource.get("Meta", {})
    links = meta.get("Link", [])

    # if Link is a object not an array 
    if isinstance(links, dict):
        links = [links]

    for link in links:
        counter +=1
        if link.get("@Rel") == "next":
            next_href = link.get("@Href")
            if next_href.startswith("https://api.lufthansa.com"):
                return next_href.replace("https://api.lufthansa.com", "")
            
        
        elif link.get("@Rel") == "last":
            last_href = link.get("@Href")
            print(f"found only last href {last_href}", file=sys.stderr)
            return None
    if counter == 4:
        return "Done"
        #if next and last is missing only 4 links are available
            #"@Href": "https://api.lufthansa.com/v1/mds-references/airports?limit=100&offset=1500",
            #"@Rel": "next"
            

    return None





def find_href(json_data: dict , meta_data_key:str) -> Optional[str]:
    
    airport_resource = json_data.get(meta_data_key, {})
    meta = airport_resource.get("Meta", {})
    links = meta.get("Link", [])

    if isinstance(links, dict):
         links = [links]

    for link in links:
        if link.get("@Rel") == "self":
              working_href = link.get("@Href")
        
        working_href.find("offset")
        if not working_href:
                print("there is no next_href")
                return None
        
def extract_offset_from_endpoint(endpoint: str) -> Optional[int]:
    parsed = urlparse(endpoint)
    qs = parse_qs(parsed.query)
    value = qs.get("offset")
    if value:
        return int(value[0])
    return None

def jump_offset(endpoint: str, skipped_values: int) -> Optional[str]:
    parsed = urlparse(endpoint)
    qs = parse_qs(parsed.query)

    limit_values = qs.get("limit")
    offset_values = qs.get("offset")

    if not limit_values or not offset_values:
        return None

    limit_value = int(limit_values[0])
    offset_value = int(offset_values[0])

    new_offset = offset_value + skipped_values

    return f"{parsed.path}?limit={limit_value}&offset={new_offset}"

def processing_Error(json_data: dict, meta_data_key: str) ->bool:
    
    #if there is not the specific meta key eg. airlineResources 
    # do a extra loop after time out 
    # if "Internal Server Error or prosssing error "
    if meta_data_key not in json_data:
        print(f"Missing meta key: {meta_data_key}", file=sys.stderr)
        save_json_locally(
                    json_data=json_data,
                    base_filename=f"{meta_data_key}error.json",
                    local_folder="errorMessages",
                    offset=0
                )
        return True
    if json_data.get(meta_data_key, {}) is None:
        return True
    #if json_data.get("ProcessingErrors", {}):
    #    return True
    return False


In [ ]:
import requests
import json
import os
from datetime import datetime

from pathlib import Path
from typing import Dict, Any, List, Optional
import uuid
import time
from urllib.parse import urlparse, parse_qs
import sys

import utilis

         

def get_data_all_Reference(
    base_Url: str,
    headers: Dict[str, str],
    catalog_name: str,
    schema_name: str,
    volume_name: str,

    mds_reference: str,

    recordLimit: int, 
    offset: int,

    save_on_Databricks:bool,
    meta_data_key =str,
    local_folder = str,
    ):
    
    language = "EN"
    dic = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{mds_reference}/"
    FileName_base = f"{dic}/{mds_reference}.json"
    endpoint = f"/v1/mds-references/{mds_reference}?limit={recordLimit}&offset={offset}"
    
    dummy_count = 0
    timeout_rounds = 0
    proxy_error = False

    while True:
        print(f"sending request {dummy_count}")
        dummy_count += 1
        print(f"{base_Url}{endpoint}")
        try:
            response = requests.get(
                base_Url + endpoint,
                headers=headers,
                timeout=10
            )

            if response.status_code in (503, 504, 429):
                timeout_rounds = utilis.timeout_api_restriction(response.status_code)
                if timeout_rounds == 5:
                    raise Exception("infinite Loop")
                continue
            
            timeout_rounds = utilis.reset_timeout_rounds(timeout_rounds)
            if response.status_code != 200:
                raise Exception(f"new error code: {response.status_code}")
            
            print("response.url =", response.url)
            json_data = response.json()
            
            if utilis.processing_Error(json_data, meta_data_key):
                if proxy_error is True:
                    raise BrokenPipeError
                time.sleep(10)
                proxy_error = True
                continue



            if(save_on_Databricks == False):
                utilis.save_json_locally(
                    json_data=json_data,
                    base_filename=f"{mds_reference}.json",
                    local_folder=local_folder,
                    offset=offset
                )
            
            if(save_on_Databricks == True ):
                utilis.save_in_Notebooks(
                    FileName_base,
                    json_data,
                    offset)
                
            #endpoint_backup = endpoint
            offset = utilis.extract_offset_from_endpoint(endpoint)
            endpoint =utilis.get_next_endpoint_from_response(json_data, meta_data_key)
            
            if endpoint == "Done":
                return f"all files successfuly saved"
            
        except Exception as e:
            print(f"{mds_reference}: api called failed \n \
                   last api endpoint {endpoint} {e}", file=sys.stderr)
            return

In [0]:
import requests
import json
import os
from datetime import datetime

from pathlib import Path
from typing import Dict, Any, List, Optional
import uuid
import time

import get_utilis
import get_airports
import get_countries
import get_airlines
import get_cities
import get_aircrafts
import get_all_Recources
import get_flight_schedules

# password = dbutils.secrets.get(scope="lh-api", key="password")
base_url = "https://lh-proxy.onrender.com"
headers ={"password": "DataIntelligence2026"}

catalog_name ="data_catalog"
schema_name = "bronze"
volume_name = "bronze_volume"

mds_reference = ["countries", "cities", "airports","airlines", "aircrafts"]
meta_data_key = ["CountryResource", "CityResource", "AirportResource","AirlineResource", "AircraftResource"]
local_folder = ["Countries", "Cities", "Airports","Airlines", "Aircrafts"]

recordLimit = 100
offset =0

for ref, key, folder in zip(mds_reference, meta_data_key, local_folder):
    get_all_Recources.get_data_all_Reference(
        base_Url="https://lh-proxy.onrender.com",
        headers=headers,
        
        catalog_name=catalog_name,
        schema_name=schema_name,
        volume_name=volume_name,
        mds_reference=ref,
        recordLimit=recordLimit,
        offset=offset,

        save_on_Databricks=False,
        meta_data_key=key,
        local_folder=folder
    )

In [0]:
#Get by recordOffset 
password = dbutils.secrets.get(scope="lh-api", key="password")

Catalog_name="data_catalog",
schema_name="bronze",
volume_name="bronze_volume",


#Reference_Types = ["mds-reference", "Offers", "Operations"]
Reference_Types = ["mds-reference", "Offers"]
mds_reference = ["Countries", "Cities", "Airports", "NearestAirport", "Aircrafts"]
Offers = [ "SeatMaps", "Lounges"]

SeatMaps = ["flightNumber", "origin", "destination", "departureDate", "cabinTypeCode"]
lounges = ["code", "cabinClassCode", "tierCode", "languageCode"]
#Operations = ["FlightSchedule","Flightstatus","" ]
#Flightstatus = ["route"]


Route= ["{Departure_Airport}","{Destination_Airport}","{date}","{serviceType}" ]

Departure_Airport = "FRA"
Destination_Airport = "LIS"
Date = "2026-03-06T10:00"
ServiceType = "passenger"
Aircraft_code="333"
Reference="mds-reference"


# 
result = get_data_Reference(
    base_Url="https://lh-proxy.onrender.com",
    headers={"password": "{password}"},
    #endpoint="/v1/references/airports/{Departure_Airport}",
    Reference_Types={Reference},
    mds_reference={mds_reference},
    Offers={Offers},


    
    Catalog_name={Catalog_name},
    schema_name={schema_name},
    volume_name={volume_name},
    Date={Date}, # from Datetime
    Departure_Airport={Departure_Airport},
    Destination_Airport="LIS",

    Aircraft_code={Aircraft_code},
)


markedown test 
create a schema 
create a catalog 
pass the data into the catalog

In [0]:
schema_name = "bronze_schema"

# full_table_name = f"filter0.{schema_name}.airport"


# # Drop the table if it exists
# spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")

# Path to the Delta data
#DATA_PATH = "/Volumes/dbacademy_wine_quality_data/v01/data"


# dbutils.fs.mkdirs("/Volumes/dblufthansa/v01/data")

DATA_PATH = "/Volumes/dblufthansa/v01/data"

# Import functions
from pyspark.sql import functions as F

full_table_name = f"filter0.{schema_name}.airport"

# Read the Delta data
df = spark.read.format("delta").load(DATA_PATH)

# Write the data to the table
df.write.mode("overwrite").saveAsTable(full_table_name)
spark.sql("CREATE CATALOG IF NOT EXISTS bronze_Catalog")
spark.sql("USE CATALOG bronze_Catalog")
schema_name = "bronze_schema"

full_table_name.show()



#